## Building the model to predict yield

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql import functions as F

from sklearn.metrics import mean_absolute_percentage_error
from xgboost.spark import SparkXGBRegressor

In [0]:
sql_query = """SELECT  cy.Year, cy.Country, cy.Item, cy.Element, cy.Unit, cy.Yield, avg_temp.Temperature, corr_index.CPI as CorruptionIndex, sog.GDP, up.UrbanPopulation, up.RuralPopulation, fu.FertilizerUse
FROM agriculture_db.crop_yield cy
INNER JOIN agriculture_db.country_codes cc ON cc.ISONum = cy.AreaCodeM49
INNER JOIN 
(SELECT AVG(ast.Temperature) as Temperature, ast.Year, ast.ISO3
FROM agriculture_db.average_surface_temperature ast
GROUP BY ast.Country, ast.Year, ast.ISO3) avg_temp ON avg_temp.ISO3 = cc.ISO3 AND avg_temp.Year = cy.Year
INNER JOIN agriculture_db.corruption_index corr_index ON corr_index.ISO = cc.ISO3 AND corr_index.Year = cy.Year
INNER JOIN agriculture_db.share_of_gdp sog ON sog.Code = cc.ISO3 AND sog.Year = cy.Year
INNER JOIN agriculture_db.urban_population up ON up.Code = cc.ISO3 AND up.Year = cy.Year
INNER JOIN agriculture_db.fertilizer_use fu ON fu.ISO3 = cc.ISO3 AND fu.Year = cy.Year"""
features_df = spark.sql(sql_query)

In [0]:
sql_query = """SELECT  cy.Year, cy.Country, cy.Item, cy.ItemCode, cy.Element, cy.Unit, cy.Yield, avg_temp.Temperature, corr_index.CPI as CorruptionIndex, sog.GDP, up.UrbanPopulation, up.RuralPopulation, fu.FertilizerUse,
peia.PeopleEmployedInAgriculture, fw.FreshwaterWithdrawal
FROM agriculture_db.crop_yield cy
INNER JOIN agriculture_db.country_codes cc ON cc.ISONum = cy.AreaCodeM49
INNER JOIN 
(SELECT AVG(ast.Temperature) as Temperature, ast.Year, ast.ISO3
FROM agriculture_db.average_surface_temperature ast
GROUP BY ast.Country, ast.Year, ast.ISO3) avg_temp ON avg_temp.ISO3 = cc.ISO3 AND avg_temp.Year = cy.Year
INNER JOIN agriculture_db.corruption_index corr_index ON corr_index.ISO = cc.ISO3 AND corr_index.Year = cy.Year
INNER JOIN agriculture_db.share_of_gdp sog ON sog.Code = cc.ISO3 AND sog.Year = cy.Year
INNER JOIN agriculture_db.urban_population up ON up.Code = cc.ISO3 AND up.Year = cy.Year
INNER JOIN agriculture_db.fertilizer_use fu ON fu.ISO3 = cc.ISO3 AND fu.Year = cy.Year
INNER JOIN agriculture_db.people_employed_in_agriculture peia ON peia.ISO3 = cc.ISO3 AND peia.Year = cy.Year
INNER JOIN agriculture_db.freshwater_withdrawal fw ON fw.ISO3 = cc.ISO3 AND fw.Year = cy.Year"""
features_df = spark.sql(sql_query)

In [0]:
crop_categories_df = spark.table("agriculture_db.crop_categories")

In [0]:
crop_categories_df.select("Category").distinct().show(50, False)

In [0]:
features_df = features_df.dropna(subset=["Yield", "CorruptionIndex", "FertilizerUse", "FreshwaterWithdrawal"])
features_df = (features_df
    .withColumn("Temperature", F.round(F.col("Temperature"), 2))
)

# filter_list = ["Sheep", "sheep", "buffalo", "Buffalo", "cattle", "Cattle", "Milk", "Eggs", "eggs", "hides", "milk"]
# filter_condition = "|".join(filter_list)
features_df = features_df.filter((F.col("Element") == "Yield") & (F.col("Yield") > 0))

In [0]:
features_df_category = features_df.join(crop_categories_df, on=["ItemCode", "Item"], how="left")
crops_df = features_df_category.filter(((F.col("Category") == "Vegetables Primary") | 
                                        (F.col("Category") == "Crops, primary") |
                                        (F.col("Category") == "Oilcrops, Oil Equivalent") |
                                        (F.col("Category") == "Citrus Fruit, Total") |
                                        (F.col("Category") == "Cereals, primary") |
                                        (F.col("Category") == "Oilcrops, Cake Equivalent") |
                                        (F.col("Category") == "Sugar Crops Primary") |
                                        (F.col("Category") == "Fruit Primary") |
                                        (F.col("Category") == "Oilcrops Primary") |
                                        (F.col("Category") == "Fibre Crops Primary") |
                                        (F.col("Category") == "Fibre Crops, Fibre Equivalent") |
                                        (F.col("Category") == "Roots and Tubers, Total"))
                                       & (F.col("Element") == "Yield"))
filter_list = ["Cucumbers", "Tomatoes", "Asparagus", "Peas", "String beans", "Broad beans", "Cassava", "Green garlic"]
filter_condition = "|".join(filter_list)
vegetables_df = features_df_category.filter(((F.col("Category") == "Vegetables Primary") |
                                             (F.col("Category") == "Roots and Tubers, Total")) 
                                            & (F.col("Element") == "Yield"))

vegetables_filtered_df = vegetables_df.filter(~F.col("Item").rlike(filter_condition))

In [0]:
pd_df = vegetables_df.select("Item", "Yield").toPandas()
pd_filtered_df = vegetables_filtered_df.select("Item", "Yield").toPandas()
pd_df = pd_df.sort_values(by="Item").reset_index(drop=True)
pd_filtered_df = pd_filtered_df.sort_values(by="Item").reset_index(drop=True)

fig, ax = plt.subplots(2, 1, figsize=(20, 15))

pd_df.boxplot(by="Item", column=["Yield"], ax=ax[0], showfliers=False)
ax[0].tick_params(axis="x", rotation=90)

pd_filtered_df.boxplot(by="Item", column=["Yield"], ax=ax[1], showfliers=False)
ax[1].tick_params(axis="x", rotation=90)

# fig.subplots_adjust(hspace=0.2)
plt.suptitle("Box Plots of Yield by Item")
plt.tight_layout()
plt.show()

In [0]:
item_indexer = StringIndexer(inputCol="Item", outputCol="ItemIndexed", handleInvalid="keep")
item_encoder = OneHotEncoder(inputCol="ItemIndexed", outputCol="ItemEncoded", dropLast=False)
assembler = VectorAssembler(inputCols=["ItemEncoded", "Temperature", "CorruptionIndex", "GDP", "UrbanPopulation", "RuralPopulation", "FertilizerUse",  "FreshwaterWithdrawal"], outputCol="features")

In [0]:
linear_regression_model = LinearRegression(featuresCol="features", labelCol="Yield")

In [0]:
pipeline = Pipeline(stages=[item_indexer, item_encoder, assembler, linear_regression_model])
model = pipeline.fit(features_df)

In [0]:
train_data, test_data = features_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
random_forest_model = RandomForestRegressor(featuresCol="features", labelCol="Yield", numTrees=100)
random_forest_pipeline = Pipeline(stages=[item_indexer, item_encoder, assembler, random_forest_model])

In [0]:
train_data, test_data = features_df.randomSplit([0.8, 0.2], seed=42)
rf_model = random_forest_pipeline.fit(train_data)
predictions = rf_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
gbt_model = GBTRegressor(featuresCol="features", labelCol="Yield", maxIter=100)
gbt_pipeline = Pipeline(stages=[item_indexer, item_encoder, assembler, gbt_model])

In [0]:
train_data, test_data = features_df.randomSplit([0.8, 0.2], seed=42)
gbt_fit_model = gbt_pipeline.fit(train_data)
predictions = gbt_fit_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
xgboost_model = SparkXGBRegressor(features_col="features", label_col="Yield")
paramGrid = (ParamGridBuilder()
    .addGrid(xgboost_model.max_depth, [3, 5, 7])
    .addGrid(xgboost_model.learning_rate, [0.05, 0.1, 0.2])
    .addGrid(xgboost_model.n_estimators, [50, 100, 200])
    .addGrid(xgboost_model.subsample, [0.8, 1.0])
    .addGrid(xgboost_model.colsample_bytree, [0.8, 1.0])
    .build())

In [0]:
evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
cross_validator = CrossValidator(estimator=xgboost_model, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=5)
xgboost_pipeline = Pipeline(stages=[item_indexer, item_encoder, assembler, cross_validator])
train_data, test_data = features_df.randomSplit([0.8, 0.2], seed=42)

In [0]:
xgboost_fit_model = xgboost_pipeline.fit(train_data)
predictions = xgboost_fit_model.transform(test_data)

In [0]:
best_model = xgboost_fit_model.stages[-1].bestModel
print(f"Best max_depth: {best_model.getOrDefault('max_depth')}")
print(f"Best learning_rate: {best_model.getOrDefault('learning_rate')}")
print(f"Best n_estimators: {best_model.getOrDefault('n_estimators')}")
print(f"Best subsample: {best_model.getOrDefault('subsample')}")
print(f"Best colsample_bytree: {best_model.getOrDefault('colsample_bytree')}")

In [0]:
xgboost_model_tuned = SparkXGBRegressor(features_col="features", label_col="Yield", max_depth=7, learning_rate=0.2, min_split_loss=1.0, n_estimators=200, subsample=0.5, colsample_bytree=0.8)
xgboost_pipeline_tuned = Pipeline(stages=[item_indexer, item_encoder, assembler, xgboost_model_tuned])

In [0]:
train_data, test_data = features_df.randomSplit([0.8, 0.2], seed=42)
xgboost_fit_model_tuned = xgboost_pipeline_tuned.fit(train_data)
predictions = xgboost_fit_model_tuned.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
true = [row["Yield"] for row in predictions.select("Yield").collect()]
pred = [row["prediction"] for row in predictions.select("prediction").collect()]
mean_absolute_percentage_error(true, pred)

In [0]:
train_data, test_data = vegetables_df.randomSplit([0.8, 0.2], seed=42)
xgboost_fit_model_tuned = xgboost_pipeline_tuned.fit(train_data)
predictions = xgboost_fit_model_tuned.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
true = [row["Yield"] for row in predictions.select("Yield").collect()]
pred = [row["prediction"] for row in predictions.select("prediction").collect()]
mean_absolute_percentage_error(true, pred)

In [0]:
agriculture_yield = features_df.groupBy("Year", "Country", "Item").agg(F.sum("Yield").alias("TotalYield"))
# agriculture_yield = features_df.join(agriculture_yield, ["Country", "Year"], "left").dropDuplicates(["Year", "Country"])

In [0]:
total_yield_assembler = VectorAssembler(inputCols=["Temperature", "CorruptionIndex", "GDP", "UrbanPopulation", "RuralPopulation", "FertilizerUse",  "FreshwaterWithdrawal"], outputCol="features")
xgboost_model_for_total = SparkXGBRegressor(features_col="features", label_col="TotalYield", max_depth=6, learning_rate=0.2, min_split_loss=1.0, n_estimators=200, subsample=0.8, colsample_bytree=0.8)
xgboost_pipeline_for_total = Pipeline(stages=[total_yield_assembler, xgboost_model_for_total])

In [0]:
train_data, test_data = agriculture_yield.randomSplit([0.8, 0.2], seed=42)
xgboost_fit_model_for_total = xgboost_pipeline_for_total.fit(train_data)
predictions = xgboost_fit_model_for_total.transform(test_data)

evaluator = RegressionEvaluator(labelCol="TotalYield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")